# Penguin Species Classification

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from penguins_project.data import validate_penguins
from penguins_project.train import (
    ASSIGNMENTS_PATH,
    DF_PARQUET_PATH,
    ID_COLUMN,
    METADATA,
    MODEL_DATA_PATH,
    MODEL_FEATURES,
    SEED,
    SPLIT_COLUMN,
    TARGET_COLUMN,
)

sns.set_theme(style="whitegrid", context="notebook")

SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_PALETTE = {
    "Adelie": "#4C78A8",
    "Chinstrap": "#F58518",
    "Gentoo": "#54A24B",
}


## 2. Load and verify the data

In [ ]:
raw_data = pd.read_parquet(DF_PARQUET_PATH)
validate_penguins(raw_data)

model_data = pd.read_parquet(MODEL_DATA_PATH)
assignments = pd.read_parquet(ASSIGNMENTS_PATH)

data_with_split = model_data.merge(
    assignments,
    on=ID_COLUMN,
    validate="one_to_one",
)

assert len(data_with_split) == 342
assert data_with_split[ID_COLUMN].is_unique
assert data_with_split[SPLIT_COLUMN].notna().all()
assert data_with_split[list(MODEL_FEATURES)].notna().all().all()

pd.crosstab(
    data_with_split[SPLIT_COLUMN],
    data_with_split[TARGET_COLUMN],
    margins=True,
)


In [ ]:
train_df = data_with_split.loc[
    data_with_split[SPLIT_COLUMN] == "train"
].copy()

validate_df = data_with_split.loc[
    data_with_split[SPLIT_COLUMN] == "validate"
].copy()

test_df = data_with_split.loc[
    data_with_split[SPLIT_COLUMN] == "test"
].copy()

print("Train:", train_df.shape)
print("Validation:", validate_df.shape)
print("Test:", test_df.shape)


## 3. Exploratory data analysis on the training split

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.countplot(
    data=train_df,
    x=TARGET_COLUMN,
    order=SPECIES_ORDER,
    color="#4C78A8",
    ax=ax,
)

ax.bar_label(ax.containers[0], padding=3)
ax.set_title("Species Distribution in the Training Set")
ax.set_xlabel("Species")
ax.set_ylabel("Number of Penguins")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=train_df,
    x="bill_length_mm",
    y="bill_depth_mm",
    hue=TARGET_COLUMN,
    style=TARGET_COLUMN,
    hue_order=SPECIES_ORDER,
    style_order=SPECIES_ORDER,
    palette=SPECIES_PALETTE,
    s=65,
    alpha=0.8,
    ax=axes[0],
)
axes[0].set_title("Bill Measurements by Species")
axes[0].set_xlabel("Bill Length (mm)")
axes[0].set_ylabel("Bill Depth (mm)")

sns.scatterplot(
    data=train_df,
    x="flipper_length_mm",
    y="body_mass_g",
    hue=TARGET_COLUMN,
    style=TARGET_COLUMN,
    hue_order=SPECIES_ORDER,
    style_order=SPECIES_ORDER,
    palette=SPECIES_PALETTE,
    s=65,
    alpha=0.8,
    ax=axes[1],
)
axes[1].set_title("Body Size Measurements by Species")
axes[1].set_xlabel("Flipper Length (mm)")
axes[1].set_ylabel("Body Mass (g)")

plt.tight_layout()
plt.show()


In [ ]:
FEATURE_LABELS = {
    "bill_length_mm": "Bill Length (mm)",
    "bill_depth_mm": "Bill Depth (mm)",
    "flipper_length_mm": "Flipper Length (mm)",
    "body_mass_g": "Body Mass (g)",
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, feature in zip(axes.flat, MODEL_FEATURES):
    sns.boxplot(
        data=train_df,
        x=TARGET_COLUMN,
        y=feature,
        order=SPECIES_ORDER,
        hue=TARGET_COLUMN,
        hue_order=SPECIES_ORDER,
        palette=SPECIES_PALETTE,
        dodge=False,
        legend=False,
        ax=ax,
    )
    ax.set_title(FEATURE_LABELS[feature])
    ax.set_xlabel("Species")
    ax.set_ylabel(FEATURE_LABELS[feature])

fig.suptitle("Feature Distributions by Penguin Species", fontsize=16)
plt.tight_layout()
plt.show()


## 4. Prepare model inputs

In [ ]:
def split_xy(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    X = frame[list(MODEL_FEATURES)].copy()
    y = frame[TARGET_COLUMN].copy()
    return X, y


X_train, y_train = split_xy(train_df)
X_validate, y_validate = split_xy(validate_df)
X_test, y_test = split_xy(test_df)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_validate.shape, y_validate.shape)
print("Test:", X_test.shape, y_test.shape)


## 5. Baseline and candidate models

The `DummyClassifier` ignores the measurements and always predicts the most frequent training class. It provides a minimum performance threshold. Logistic Regression is a scaled linear baseline, while Random Forest captures nonlinear interactions without requiring scaling.


In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1_000,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
}

validation_rows = []
fitted_models = {}

for model_name, model in models.items():
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)
    predictions = fitted_model.predict(X_validate)

    validation_rows.append(
        {
            "model": model_name,
            "accuracy": accuracy_score(y_validate, predictions),
            "macro_f1": f1_score(
                y_validate,
                predictions,
                average="macro",
            ),
        }
    )
    fitted_models[model_name] = fitted_model

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

validation_results.style.format(
    {"accuracy": "{:.3f}", "macro_f1": "{:.3f}"}
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

plot_results = validation_results.melt(
    id_vars="model",
    value_vars=["accuracy", "macro_f1"],
    var_name="metric",
    value_name="score",
)

sns.barplot(
    data=plot_results,
    x="score",
    y="model",
    hue="metric",
    palette={"accuracy": "#4C78A8", "macro_f1": "#F58518"},
    ax=ax,
)

ax.set_xlim(0, 1)
ax.set_title("Validation Performance")
ax.set_xlabel("Score")
ax.set_ylabel("Model")
ax.legend(title="Metric")

plt.tight_layout()
plt.show()


## 6. Select the model and evaluate the untouched test split

In [ ]:
candidate_results = validation_results.loc[
    validation_results["model"] != "Dummy baseline"
]

best_model_name = candidate_results.iloc[0]["model"]
best_validation_macro_f1 = candidate_results.iloc[0]["macro_f1"]

development_df = pd.concat(
    [train_df, validate_df],
    ignore_index=True,
)
X_development, y_development = split_xy(development_df)

final_model = clone(models[best_model_name])
final_model.fit(X_development, y_development)

test_predictions = final_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
)

print("Selected model:", best_model_name)
print(f"Validation macro F1: {best_validation_macro_f1:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test macro F1: {test_macro_f1:.3f}")


In [ ]:
classification_report_df = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=SPECIES_ORDER,
        output_dict=True,
        zero_division=0,
    )
).T

classification_report_df.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    labels=SPECIES_ORDER,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)

ax.set_title(f"Test Confusion Matrix: {best_model_name}")
plt.tight_layout()
plt.show()


## 7. Error analysis

In [ ]:
error_columns = [
    ID_COLUMN,
    TARGET_COLUMN,
    *MODEL_FEATURES,
    *METADATA,
]

test_results = test_df[error_columns].copy()
test_results["predicted_species"] = test_predictions
test_results["correct"] = (
    test_results[TARGET_COLUMN] == test_results["predicted_species"]
)

errors = test_results.loc[~test_results["correct"]].copy()

print(f"Test errors: {len(errors)} of {len(test_results)}")
errors


In [ ]:
def subgroup_summary(
    frame: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    summary = (
        frame.groupby(group_column, dropna=False)
        .agg(
            observations=("correct", "size"),
            correct_predictions=("correct", "sum"),
            accuracy=("correct", "mean"),
        )
        .reset_index()
    )
    summary["errors"] = (
        summary["observations"] - summary["correct_predictions"]
    )
    return summary


print("Accuracy by island")
display(subgroup_summary(test_results, "island"))

print("Accuracy by sex (metadata only)")
display(subgroup_summary(test_results, "sex"))


Subgroup results are descriptive only because the test set is small. `island` and `sex` were not used as predictors; they are inspected only to identify concentrated errors.


## 8. Model-agnostic feature importance

In [ ]:
permutation_result = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=30,
    random_state=SEED,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame(
        {
            "feature": MODEL_FEATURES,
            "importance_mean": permutation_result.importances_mean,
            "importance_std": permutation_result.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

importance_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ordered_importance = importance_df.sort_values("importance_mean")

ax.barh(
    ordered_importance["feature"],
    ordered_importance["importance_mean"],
    xerr=ordered_importance["importance_std"],
    color="#4C78A8",
    alpha=0.85,
)

ax.axvline(0, color="black", linewidth=1)
ax.set_title("Permutation Importance on the Test Split")
ax.set_xlabel("Decrease in Macro F1 After Permutation")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()


## 9. Conclusions

In [ ]:
error_count = len(errors)
most_important_feature = importance_df.iloc[0]["feature"]

display(
    Markdown(
        f'''
### Final result

- **Selected model:** {best_model_name}
- **Validation macro F1 before final refit:** {best_validation_macro_f1:.3f}
- **Test accuracy:** {test_accuracy:.3f}
- **Test macro F1:** {test_macro_f1:.3f}
- **Test errors:** {error_count} of {len(test_results)} observations
- **Largest permutation-importance estimate:** `{most_important_feature}`

The selected model substantially improves on the most-frequent-class baseline. Physical measurements contain enough information to distinguish the three species without using island or sex as predictors. Remaining mistakes should be interpreted cautiously because the test set contains only {len(test_results)} observations.
'''
    )
)
